In [115]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [116]:
# Nhập thư viện Matplotlib và Seaborn để trực quan hóa dữ liệu
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [117]:
# Nhập bộ dataset Titanic trong folder data có tên là 'titanic.csv'
from google.colab import drive
drive.mount('/content/drive')

train_df = pd.read_csv('/content/drive/MyDrive/Machine_learning/learn_EDA/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Machine_learning/learn_EDA/test.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [118]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [119]:
def display_missing_data(df):
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    percent = (missing / len(df)) * 100
    print(pd.DataFrame({'Missing Values': missing, 'Percent (%)': percent.round(2)}))

display_missing_data(train_df)

          Missing Values  Percent (%)
Cabin                687        77.10
Age                  177        19.87
Embarked               2         0.22


In [120]:
display_missing_data(test_df)

       Missing Values  Percent (%)
Cabin             327        78.23
Age                86        20.57
Fare                1         0.24


In [121]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [122]:
# Feature engineering
def extract_title(name):
    m = re.search(r',\s*([^\.]+)\.', name)
    return m.group(1).strip() if m else ''

title_map = {
    'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Dr': 'Officer', 'Rev': 'Officer',
    'Don': 'Royalty', 'Sir': 'Royalty', 'Lady': 'Royalty', 'the Countess': 'Royalty', 'Jonkheer': 'Royalty', 'Dona': 'Royalty'
}

def get_surname(name):
    return name.split(", ")[0]
for df in [train_df, test_df]:
    # --- Title (Name) ---
    df['Title'] = df['Name'].apply(extract_title)
    df['Title'] = df['Title'].replace(['Mlle','Ms'],'Miss')
    df['Title'] = df['Title'].replace(['Mme'],'Mrs')
    df['Title'] = df['Title'].replace(title_map)
    rare = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
    df['Title'] = df['Title'].replace(list(rare), 'Other')

    # Họ
    # Tạo một cột mới chứa họ của các hành khách


# Dùng lệnh apply để tạo một cột mới
    df["Surname"] = df["Name"].apply(get_surname)
    # --- Sex ---
    df['IsFemale'] = (df['Sex'] == 'female').astype(int)

    # --- Family (SibSp/Parch) ---
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # --- Special groups ---
    df['IsChild'] = (df['Age'] < 12).astype(int)
    df['IsMother'] = ((df['Sex'] == 'female') & (df['Parch'] > 0) & (df['Age'] > 18) & (df['Title'] == 'Mrs')).astype(int)

In [123]:
# Fill missing values
for df in [df, test_df]:
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))

    df['Age'] = df.groupby(['Sex','Pclass','Title'])['Age'].transform(lambda x: x.fillna(x.median()))

    # --- Chuẩn hóa log ---
    df['Fare'] = np.log1p(df['Fare'])
    df['Age'] = np.log1p(df['Age'])

display_missing_data(df)
display_missing_data(test_df)

       Missing Values  Percent (%)
Cabin             327        78.23
       Missing Values  Percent (%)
Cabin             327        78.23


In [124]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
 12  Title        891 non-null    object 
 13  Surname      891 non-null    object 
 14  IsFemale     891 non-null    int64  
 15  FamilySize   891 non-null    int64  
 16  IsChild      891 non-null    int64  
 17  IsMother     891 non-null    int64  
dtypes: float64(2), int64(9), object(7)
memory usage: 1

In [125]:
for df in [train_df, test_df]:

    df['HasCabin'] = df['Cabin'].notna().astype(int)

    df['Age*Pclass'] = df['Age'] * df['Pclass']
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    # df['Embarked_Pclass'] = df['Embarked'] + "_" + df['Pclass'].astype(str)
    df['FamilyCategory'] = pd.cut(df['FamilySize'], bins=[0,1,4,11], labels=['Single','Small','Large'])
    df['AgeBin'] = pd.cut(df['Age'], bins=[0,12,20,40,60,80], labels=['Child','Teen','Adult','MidAge','Senior'])
    # --- Cabin / Deck ---
    df['Deck'] = df['Cabin'].astype(str).str[0]
    df['Deck'] = df['Deck'].replace('n', 'U')  # 'U' = Unknown

    # --- Interaction features ---
    df['Fare_Pclass'] = df['Fare'] / df['Pclass']
    df['Age*Pclass'] = df['Age'] * df['Pclass']

    df['TicketPrefix'] = df['Ticket'].apply(lambda x: re.split(r'\s|\.', str(x))[0])
    df['TicketPrefix'] = df['TicketPrefix'].apply(lambda x: x if x.isalpha() else 'NUM')

In [126]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   PassengerId     891 non-null    int64   
 1   Survived        891 non-null    int64   
 2   Pclass          891 non-null    int64   
 3   Name            891 non-null    object  
 4   Sex             891 non-null    object  
 5   Age             714 non-null    float64 
 6   SibSp           891 non-null    int64   
 7   Parch           891 non-null    int64   
 8   Ticket          891 non-null    object  
 9   Fare            891 non-null    float64 
 10  Cabin           204 non-null    object  
 11  Embarked        889 non-null    object  
 12  Title           891 non-null    object  
 13  Surname         891 non-null    object  
 14  IsFemale        891 non-null    int64   
 15  FamilySize      891 non-null    int64   
 16  IsChild         891 non-null    int64   
 17  IsMother        

## Chia nhỏ dữ liệu

In [127]:
# Prepare features
feature_cols = [
    'Pclass','Sex','Age','Fare','Embarked','Title','FamilySize',
    'IsChild','IsMother','Deck','HasCabin','Fare_Pclass','Age*Pclass','TicketPrefix','IsAlone'
]


x = train_df[feature_cols]
y = train_df['Survived']
x_test_final = test_df[feature_cols]
pd.concat([train_df['Title'], test_df['Title']]).value_counts()


,count
Title,
Mr,757
Miss,264
Mrs,198
Master,61
Officer,18
Other,11


In [128]:
for col in feature_cols:
    if col not in train_df.columns:
        print(f"Train missing: {col}")
    if col not in test_df.columns:
        print(f"Test missing: {col}")


### Preprocess pipeline

In [129]:
# Preprocessor (ColumnTransformer)
num_features = [
    'Age','Fare','FamilySize','Fare_Pclass','Age*Pclass'
]
cat_features = [
    'Pclass','Sex','Embarked','Title','IsChild','IsMother',
    'Deck','HasCabin','TicketPrefix', 'IsAlone'
]

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_transformer, num_features), ('cat', cat_transformer, cat_features)])

In [130]:
df_encoded = pd.get_dummies(df, columns=[
    'Pclass','Sex','Embarked','Title','Deck','TicketPrefix'
], drop_first=True)

In [131]:

scaler = StandardScaler()
df_encoded[['Age','Fare','FamilySize','Age*Pclass','Fare_Pclass']] = scaler.fit_transform(
    df_encoded[['Age','Fare','FamilySize','Age*Pclass','Fare_Pclass']]
)

In [132]:
## Models Training

In [133]:
# --- Khởi tạo các pipeline ---
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=2000,random_state=42,C=0.8,solver='lbfgs'))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=800, max_depth=6,min_samples_split=4, min_samples_leaf=2))
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1,n_estimators=600,max_depth=4,learning_rate=0.05,subsample=0.9,colsample_bytree=0.9))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ])
}

In [136]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

best_score = 0
best_model_name = None
best_model = None

for name, model in models.items():
    acc = cross_val_score(model, x, y, cv=cv, scoring='accuracy').mean()
    f1 = cross_val_score(model, x, y, cv=cv, scoring='f1').mean()
    auc = cross_val_score(model, x, y, cv=cv, scoring='roc_auc').mean()

    print(f"{name} CV Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC:  {auc:.4f}")
    print("-" * 35)


    # if acc > best_score:
    #     best_score = acc
    #     best_model_name = name
    #     best_model = model

print(f"Best model: {best_model_name} với Accuracy = {best_score:.4f}")

# Train model tốt nhất trên toàn bộ dataset


Logistic Regression CV Results:
  Accuracy: 0.8192
  F1 Score: 0.7576
  ROC AUC:  0.8704
-----------------------------------
Random Forest CV Results:
  Accuracy: 0.8305
  F1 Score: 0.7623
  ROC AUC:  0.8706
-----------------------------------
XGBoost CV Results:
  Accuracy: 0.8349
  F1 Score: 0.7756
  ROC AUC:  0.8819
-----------------------------------
SVM CV Results:
  Accuracy: 0.8316
  F1 Score: 0.7642
  ROC AUC:  0.8642
-----------------------------------
Best model: None với Accuracy = 0.0000


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'FamilySize',
                                                   'Fare_Pclass',
                                                   'Age*Pclass']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex', 'Embarked',
                                                   'Title', 'IsChild',
                                                   'IsMother', 'Deck',
                                                   'HasCabin', 'TicketPrefix',
                                                   'IsAlone'])])),
                ('model',
                 RandomForestClassifier(max_depth=6, min_samples_leaf=2,
                                        min_samples_split=4, n_estimators=800,
                                        n_jobs=-1, random_state=42))])

In [140]:
# Prepare submission (predict on test)
best_model=models['SVM']
best_model.fit(x, y)
try:
    preds = best_model.predict(x_test_final)
    submission = pd.DataFrame({
        'PassengerId': test_df['PassengerId'],
        'Survived': preds.astype(int)
    })
    submission.to_csv('/content/drive/MyDrive/Machine_learning/learn_EDA/submissionnn.csv', index=False)
    print("File 'submission.csv' created.")
except Exception as e:
    print('Không thể tạo submission tự động:', e)


File 'submission.csv' created.
